# Passo 1 — differenze di stato medio SENS − CTRL (figure)

Legge i `.nc` di `01-mean_state_delta` e salva le figure in
`FIG_DIR/cover_mechanism/`:

1. **Mappe** delle differenze (vegetazione, albedo, neve, tas) per stagione, a
   un lead fissato. La `x` marca i punti non significativi (p ≥ 0.05); la
   vegetazione è prescritta e non ha test.
2. **Relazioni tra pixel**: per classi di Δx, la media di Δy (pesata con cos lat)
   con l'intervallo bootstrap al 95% sugli anni, separatamente per pixel con e
   senza neve (o per fascia di latitudine), sulle terre e nel box Siberia. Le
   catene da controllare:
   - Δcvh, Δcvl → Δalbedo (inverno e primavera: la vegetazione alta copre la neve);
   - Δalbedo → Δtas;
   - Δcvh → Δtas (effetto complessivo);
   - ΔLAI → Δtas (estate: evapotraspirazione).
3. **Coerenza tra lead**: se la causa è la vegetazione prescritta, la
   differenza deve essere simile a tutti i lead.

Come leggere le pendenze: sono sensibilità del modello stimate confrontando
pixel diversi (spazio al posto del tempo), dentro un'unica coppia di
esperimenti. L'intervallo bootstrap dice quanto la relazione dipende dagli anni
e dalla variabilità interna, non tiene conto della correlazione tra pixel
vicini. Anche altre differenze tra SENS e CTRL che variano insieme alla cover
(per esempio i tipi di vegetazione) possono contribuire.

In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import WORK_DIR, FIG_DIR

LEAD = 1            # lead per mappe e relazioni (1 = secondo anno: evita il primo inverno dopo l'inizializzazione)
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
SNOW_THR = 0.01     # m di snd: soglia per "pixel con neve"
N_BOOT = 1000       # iterazioni bootstrap per gli intervalli

OUT_DIR = WORK_DIR / 'cover_mechanism'
FIG_OUT = FIG_DIR / 'cover_mechanism'
FIG_OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
sys.path.insert(0, os.getcwd())
import mechanism_lib as ml
import pandas as pd
from IPython.display import Image, display

## 1. Mappe delle differenze

Una mappa per variabile e stagione (stesso stile delle altre figure, `af.map_plot`)
e un composito per stagione.

In [ ]:
MAP_VARS = ['cvh', 'cvl', 'lai_hv', 'lai_lv', 'alb', 'snd', 'tas']
for season in SEASONS:
    pngs = ml.plot_maps(OUT_DIR, season, LEAD, MAP_VARS, FIG_OUT)
    out = ml.composite(pngs, FIG_OUT / f'mech_maps_{season}_lead{LEAD}_1999.png',
                       title=f'SENS \u2212 CTRL, {season}, lead {LEAD}')
    if out:
        display(Image(filename=str(out), width=1400))

## 2. Relazioni tra pixel

Ogni riga di `RELATIONS` è (x, y, strati): `'neve'` separa i pixel con e senza
neve (snd medio > `SNOW_THR`), `'lat'` le fasce di latitudine. La tabella
riassume pendenza [IC 95%], r di Pearson e rho di Spearman per ogni pannello.

In [ ]:
RELATIONS = [
    ('cvh', 'alb', 'neve'),      # mascheramento della neve
    ('cvl', 'alb', 'neve'),
    ('alb', 'tas', 'neve'),      # albedo -> temperatura
    ('cvh', 'tas', 'neve'),      # effetto complessivo
    ('lai_hv', 'tas', 'lat'),    # evapotraspirazione (estate)
    ('lai_lv', 'tas', 'lat'),
]
tables = []
for x, y, kind in RELATIONS:
    png = FIG_OUT / f'mech_rel_{y}_vs_{x}_lead{LEAD}_1999.png'
    tables.append(ml.plot_relation(OUT_DIR, x, y, kind, LEAD, png, seasons=SEASONS,
                                   snow_thr=SNOW_THR, n_boot=N_BOOT))
    display(Image(filename=str(png), width=1400))

table = pd.concat(tables, ignore_index=True)
table.to_csv(FIG_OUT / f'mech_relazioni_lead{LEAD}_1999.csv', index=False)
table.round(4)

## 3. Coerenza tra lead

Media d'area di Δalb, Δsnd e Δtas per stagione e lead, nel box Siberia e
nelle terre boreali, con intervallo bootstrap al 95% sugli anni.

In [ ]:
png = FIG_OUT / 'mech_lead_consistency_1999.png'
lead_table = ml.plot_lead_consistency(OUT_DIR, png, seasons=SEASONS, n_boot=N_BOOT)
lead_table.to_csv(FIG_OUT / 'mech_lead_consistency_1999.csv', index=False)
display(Image(filename=str(png), width=1100))